# NRMS Training for MIND Dataset
Neural News Recommendation with Multi-Head Self-Attention

Based on: https://github.com/fabulosa/NRMS_new-for-MIND

## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
import torch.utils.data as Data
import pickle
import random
import math
import os
from ast import literal_eval
import matplotlib.pyplot as plt
from tqdm import tqdm

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## 2. Configuration

In [2]:
# Paths
BASE_DIR = '/kaggle/input/nrms-mind/'
TRAIN_DIR = os.path.join(BASE_DIR, 'MINDsmall_train', 'MINDsmall_train')
DEV_DIR = os.path.join(BASE_DIR, 'MINDsmall_dev', 'MINDsmall_dev')
GLOVE_DIR = os.path.join(BASE_DIR, 'glove.840B.300d.txt')
OUTPUT_DIR = "/kaggle/working/preprocessed_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Hyperparameters
config = {
    'num_head_text': 8,
    'num_head_entity': 4,
    'text_attn_vector_size': 200,
    'entity_attn_vector_size': 64,
    'news_final_attn_vector_size': 48,
    'news_final_embed_size': 50,
    'history_num_head': 10,
    'history_attn_vector_size': 32,
    'recent_num_head': 10,
    'recent_attn_vector_size': 32,
    'final_attn_vector_size': 32,
    'batch_size': 64,
    'weight_decay': 1e-5,
    'negative_sampling_num': 4,  # K=4 as per paper
    'max_history_len': 50,
    'max_recent_len': 10,
    'learning_rate': 5e-4,
    'epochs': 40
}
print('Configuration loaded')

Configuration loaded


## 3. Data Preprocessing
### 3.1 GloVe Dictionary

In [3]:
def load_glove_model(file_path):
    print('Loading GloVe Model...')
    glove = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in tqdm(f, desc='Loading GloVe'):
            parts = line.split(' ')
            word = parts[0]
            embedding = np.array([float(x) for x in parts[1:]])
            glove[word] = embedding
    print(f'{len(glove)} words loaded!')
    return glove

glove_path = os.path.join(GLOVE_DIR)
glove_dict_path = os.path.join(OUTPUT_DIR, 'glove_dict.pkl')

if os.path.exists(glove_dict_path):
    print('Loading cached GloVe dictionary...')
    with open(glove_dict_path, 'rb') as f:
        glove = pickle.load(f)
else:
    glove = load_glove_model(glove_path)
    with open(glove_dict_path, 'wb') as f:
        pickle.dump(glove, f)
    print('GloVe dictionary saved')

Loading GloVe Model...


Loading GloVe: 2196017it [02:53, 12629.21it/s]


2196016 words loaded!
GloVe dictionary saved


### 3.2 News Preprocessing

In [4]:
def load_news_data():
    cols = ['NewsID', 'Category', 'SubCategory', 'Title', 'Abstract', 'URL', 'TitleEntities', 'AbstractEntities']
    news_train = pd.read_csv(os.path.join(TRAIN_DIR, 'news.tsv'), sep='\t', header=None, names=cols)
    news_dev = pd.read_csv(os.path.join(DEV_DIR, 'news.tsv'), sep='\t', header=None, names=cols)
    news = pd.concat([news_train, news_dev]).drop_duplicates()
    print(f'Loaded {len(news)} unique news articles')
    return news

news_data = load_news_data()

Loaded 65238 unique news articles


In [ ]:
def process_categories(news):
    categories = news['Category'].dropna().unique()
    subcategories = news['SubCategory'].dropna().unique()

    cat2id = {c: i for i, c in enumerate(categories)}
    subcat2id = {c: i for i, c in enumerate(subcategories)}

    newsID_catID = {row['NewsID']: cat2id.get(row['Category'], 0) for _, row in news.iterrows()}
    newsID_subcatID = {row['NewsID']: subcat2id.get(row['SubCategory'], 0) for _, row in news.iterrows()}

    return cat2id, subcat2id, newsID_catID, newsID_subcatID, len(categories), len(subcategories)

cat2id, subcat2id, newsID_catID, newsID_subcatID, num_cat, num_subcat = process_categories(news_data)
print(f'Categories: {num_cat}, SubCategories: {num_subcat}')

Categories: 18, SubCategories: 270


In [ ]:
def process_text_embeddings(news, column, glove_dict):
    all_words = set()
    news_words = {}

    for _, row in news.iterrows():
        text = str(row[column]) if pd.notna(row[column]) else ''
        words = text.lower().split()
        words = [w.strip('.,!?"\'') for w in words]
        words = [w for w in words if w in glove_dict]
        all_words.update(words)
        news_words[row['NewsID']] = words

    word_list = list(all_words)
    word2id = {w: i+1 for i, w in enumerate(word_list)}  # 0 reserved for padding

    embed_matrix = np.zeros((len(word_list)+1, 300))
    for w, i in word2id.items():
        embed_matrix[i] = glove_dict[w]

    newsID_wordIDs = {nid: [word2id[w] for w in words] for nid, words in news_words.items()}

    return embed_matrix, newsID_wordIDs

print('Processing title embeddings...')
title_embed_matrix, newsID_titleWordID = process_text_embeddings(news_data, 'Title', glove)
print(f'Title embedding matrix: {title_embed_matrix.shape}')

print('Processing abstract embeddings...')
abstract_embed_matrix, newsID_abstractWordID = process_text_embeddings(news_data, 'Abstract', glove)
print(f'Abstract embedding matrix: {abstract_embed_matrix.shape}')

Processing title embeddings...
Title embedding matrix: (33225, 300)
Processing abstract embeddings...
Abstract embedding matrix: (54087, 300)


In [ ]:
def process_entities(news, train_dir, dev_dir):
    entity_files = [
        os.path.join(train_dir, 'entity_embedding.vec'),
        os.path.join(dev_dir, 'entity_embedding.vec')
    ]

    entities = {}
    for ef in entity_files:
        if os.path.exists(ef):
            with open(ef, 'r', encoding='utf-8') as f:
                for line in f:
                    parts = line.strip().split('\t')
                    if len(parts) > 1:
                        eid = parts[0]
                        vec = [float(x) for x in parts[1:]]
                        entities[eid] = vec

    entity_list = list(entities.keys())
    entity2id = {e: i+1 for i, e in enumerate(entity_list)}

    embed_dim = len(next(iter(entities.values()))) if entities else 100
    entity_embed = np.zeros((len(entity_list)+1, embed_dim))
    for e, i in entity2id.items():
        entity_embed[i] = entities[e]

    def parse_entities(row, col):
        try:
            ents = literal_eval(row[col]) if pd.notna(row[col]) else []
            return [[entity2id.get(e['WikidataId'], 0), float(e.get('Confidence', 1.0))]
                    for e in ents if e.get('WikidataId') in entity2id]
        except:
            return []

    newsID_titleEntity = {row['NewsID']: parse_entities(row, 'TitleEntities') for _, row in news.iterrows()}
    newsID_abstractEntity = {row['NewsID']: parse_entities(row, 'AbstractEntities') for _, row in news.iterrows()}

    return entity_embed, newsID_titleEntity, newsID_abstractEntity

print('Processing entity embeddings...')
entity_embed_matrix, newsID_titleEntity, newsID_abstractEntity = process_entities(news_data, TRAIN_DIR, DEV_DIR)
print(f'Entity embedding matrix: {entity_embed_matrix.shape}')

Processing entity embeddings...
Entity embedding matrix: (31452, 100)


### 3.3 Behavior Preprocessing

In [8]:
def load_behaviors(file_path):
    cols = ['ImpressionID', 'UserID', 'Time', 'Histories', 'Impressions']
    df = pd.read_csv(file_path, sep='\t', header=None, names=cols)
    df['Time'] = pd.to_datetime(df['Time'], format='%m/%d/%Y %I:%M:%S %p')
    return df

behaviors_train = load_behaviors(os.path.join(TRAIN_DIR, 'behaviors.tsv'))
behaviors_dev = load_behaviors(os.path.join(DEV_DIR, 'behaviors.tsv'))
print(f'Train behaviors: {len(behaviors_train)}, Dev behaviors: {len(behaviors_dev)}')

Train behaviors: 156965, Dev behaviors: 73152


In [ ]:
def preprocess_behaviors(behaviors, neg_sample_num=4):
    """K=4 negative sampling for training (as per NRMS paper)."""
    samples = []

    for _, row in tqdm(behaviors.iterrows(), total=len(behaviors), desc='Processing behaviors'):
        history = str(row['Histories']).split() if pd.notna(row['Histories']) else []
        impressions = str(row['Impressions']).split() if pd.notna(row['Impressions']) else []

        pos_news = [imp.split('-')[0] for imp in impressions if imp.endswith('-1')]
        neg_news = [imp.split('-')[0] for imp in impressions if imp.endswith('-0')]

        for pos in pos_news:
            if len(neg_news) >= neg_sample_num:
                negs = random.sample(neg_news, neg_sample_num)
            elif len(neg_news) > 0:
                negs = neg_news + list(np.random.choice(neg_news, neg_sample_num - len(neg_news)))
            else:
                continue

            candidates = [pos] + negs  # First is positive, rest are negatives
            samples.append([row['UserID'], history[-config['max_history_len']:], [], candidates])

    random.shuffle(samples)
    return samples

print('Processing training behaviors (K=4 sampling)...')
training_data = preprocess_behaviors(behaviors_train, config['negative_sampling_num'])
print(f'Training samples: {len(training_data)}')

print('Processing validation behaviors (K=4 sampling)...')
validation_data = preprocess_behaviors(behaviors_dev, config['negative_sampling_num'])
print(f'Validation samples: {len(validation_data)}')

Processing training behaviors (K=4 sampling)...


Processing behaviors: 100%|██████████| 156965/156965 [00:14<00:00, 10861.68it/s]


Training samples: 236344
Processing validation behaviors (K=4 sampling)...


Processing behaviors: 100%|██████████| 73152/73152 [00:06<00:00, 11529.12it/s]


Validation samples: 111383


## 4. Model Architecture

In [10]:
def attention(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    p_attn = F.softmax(scores, dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn


class MultiHeadedAttention(nn.Module):
    def __init__(self, h, d_model, dropout=0.2):
        super().__init__()
        self.d_k = d_model // h
        self.h = h
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask=None):
        if mask is not None:
            mask = mask.unsqueeze(1)
        nbatches = query.size(0)
        query, key, value = [x.view(nbatches, -1, self.h, self.d_k).transpose(1, 2) for x in (query, key, value)]
        x, _ = attention(query, key, value, mask=mask, dropout=self.dropout)
        x = x.transpose(1, 2).contiguous().view(nbatches, -1, self.h * self.d_k)
        return x

In [ ]:
class TextEncoder(nn.Module):
    def __init__(self, word_embed_size, num_head, attn_vector_size):
        super().__init__()
        self.self_attn_size = word_embed_size // num_head * num_head

        self.weight_query = nn.Parameter(torch.FloatTensor(word_embed_size, self.self_attn_size))
        self.weight_key = nn.Parameter(torch.FloatTensor(word_embed_size, self.self_attn_size))
        self.weight_value = nn.Parameter(torch.FloatTensor(word_embed_size, self.self_attn_size))
        self.trans_weight_v = nn.Parameter(torch.FloatTensor(self.self_attn_size, attn_vector_size))
        self.trans_weight_q = nn.Parameter(torch.FloatTensor(attn_vector_size, 1))

        self.self_attn = MultiHeadedAttention(num_head, word_embed_size)
        self._init_weights()

    def _init_weights(self):
        for p in [self.weight_query, self.weight_key, self.weight_value, self.trans_weight_v, self.trans_weight_q]:
            nn.init.uniform_(p, -0.1, 0.1)

    def forward(self, embeddings, mask_selfattn, mask_attn):
        q = torch.matmul(embeddings, self.weight_query)
        k = torch.matmul(embeddings, self.weight_key)
        v = torch.matmul(embeddings, self.weight_value)

        x = self.self_attn(q, k, v, mask_selfattn)
        scores = torch.matmul(torch.tanh(torch.matmul(x, self.trans_weight_v)), self.trans_weight_q).squeeze(-1)
        scores = scores.masked_fill(mask_attn == 0, -1e9)
        attend = F.softmax(scores, dim=1)
        attend = attend.masked_fill(mask_attn.sum(dim=1, keepdim=True) == 0, 0)
        embed = torch.bmm(x.transpose(1, 2), attend.unsqueeze(-1)).squeeze(-1)
        return embed

In [ ]:
class NewsEncoder(nn.Module):
    def __init__(self, num_cat, num_subcat, title_embed, abstract_embed, entity_embed, cfg):
        super().__init__()
        self.final_embed_size = cfg['news_final_embed_size']

        self.cat_embed = nn.Embedding(num_cat + 1, self.final_embed_size)
        self.subcat_embed = nn.Embedding(num_subcat + 1, self.final_embed_size)

        self.title_word_embed = nn.Embedding.from_pretrained(torch.FloatTensor(title_embed), freeze=False)
        self.abstract_word_embed = nn.Embedding.from_pretrained(torch.FloatTensor(abstract_embed), freeze=False)
        self.entity_embed = nn.Embedding.from_pretrained(torch.FloatTensor(entity_embed), freeze=True)

        text_size = title_embed.shape[1] // cfg['num_head_text'] * cfg['num_head_text']
        entity_size = entity_embed.shape[1] // cfg['num_head_entity'] * cfg['num_head_entity']

        self.title_encoder = TextEncoder(title_embed.shape[1], cfg['num_head_text'], cfg['text_attn_vector_size'])
        self.abstract_encoder = TextEncoder(abstract_embed.shape[1], cfg['num_head_text'], cfg['text_attn_vector_size'])
        self.title_entity_encoder = TextEncoder(entity_embed.shape[1], cfg['num_head_entity'], cfg['entity_attn_vector_size'])
        self.abstract_entity_encoder = TextEncoder(entity_embed.shape[1], cfg['num_head_entity'], cfg['entity_attn_vector_size'])

        self.title_linear = nn.Linear(text_size, self.final_embed_size)
        self.abstract_linear = nn.Linear(text_size, self.final_embed_size)
        self.title_entity_linear = nn.Linear(entity_size, self.final_embed_size)
        self.abstract_entity_linear = nn.Linear(entity_size, self.final_embed_size)

        self.trans_weight_v = nn.Parameter(torch.FloatTensor(self.final_embed_size, cfg['news_final_attn_vector_size']))
        self.trans_weight_q = nn.Parameter(torch.FloatTensor(cfg['news_final_attn_vector_size'], 1))
        nn.init.uniform_(self.trans_weight_v, -0.1, 0.1)
        nn.init.uniform_(self.trans_weight_q, -0.1, 0.1)

    def forward(self, cat, subcat, title, title_mask, abstract, abstract_mask, title_ent, title_ent_mask, abstract_ent, abstract_ent_mask):
        cat_emb = self.cat_embed(cat).unsqueeze(1)
        subcat_emb = self.subcat_embed(subcat).unsqueeze(1)

        title_emb = self.title_word_embed(title)
        title_attn = self.title_encoder(title_emb, title_mask.unsqueeze(1) * title_mask.unsqueeze(2), title_mask)
        title_attn = self.title_linear(title_attn).unsqueeze(1)

        abstract_emb = self.abstract_word_embed(abstract)
        abstract_attn = self.abstract_encoder(abstract_emb, abstract_mask.unsqueeze(1) * abstract_mask.unsqueeze(2), abstract_mask)
        abstract_attn = self.abstract_linear(abstract_attn).unsqueeze(1)

        title_ent_emb = self.entity_embed(title_ent[:, :, 0].long())
        title_ent_attn = self.title_entity_encoder(title_ent_emb, title_ent_mask.unsqueeze(1) * title_ent_mask.unsqueeze(2), title_ent_mask)
        title_ent_attn = self.title_entity_linear(title_ent_attn).unsqueeze(1)

        abstract_ent_emb = self.entity_embed(abstract_ent[:, :, 0].long())
        abstract_ent_attn = self.abstract_entity_encoder(abstract_ent_emb, abstract_ent_mask.unsqueeze(1) * abstract_ent_mask.unsqueeze(2), abstract_ent_mask)
        abstract_ent_attn = self.abstract_entity_linear(abstract_ent_attn).unsqueeze(1)

        concat = torch.cat([cat_emb, subcat_emb, title_attn, abstract_attn, title_ent_attn, abstract_ent_attn], dim=1)
        attend = F.softmax(torch.matmul(torch.tanh(torch.matmul(concat, self.trans_weight_v)), self.trans_weight_q), dim=1)
        news_embed = torch.bmm(concat.transpose(1, 2), attend).squeeze(-1)
        return news_embed

In [ ]:
class NRMS(nn.Module):
    def __init__(self, news_encoder, cfg):
        super().__init__()
        self.news_encoder = news_encoder
        self.history_encoder = TextEncoder(cfg['news_final_embed_size'], cfg['history_num_head'], cfg['history_attn_vector_size'])
        self.recent_encoder = TextEncoder(cfg['news_final_embed_size'], cfg['recent_num_head'], cfg['recent_attn_vector_size'])

        embed_size = cfg['news_final_embed_size'] // cfg['history_num_head'] * cfg['history_num_head']
        self.trans_weight_v = nn.Parameter(torch.FloatTensor(embed_size, cfg['final_attn_vector_size']))
        self.trans_weight_q = nn.Parameter(torch.FloatTensor(cfg['final_attn_vector_size'], 1))
        nn.init.uniform_(self.trans_weight_v, -0.1, 0.1)
        nn.init.uniform_(self.trans_weight_q, -0.1, 0.1)

        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, history_news_embed, history_mask, candidate_news_embed):
        history_mask_2d = history_mask.unsqueeze(1) * history_mask.unsqueeze(2)
        user_embed = self.history_encoder(history_news_embed, history_mask_2d, history_mask)

        scores = torch.bmm(candidate_news_embed, user_embed.unsqueeze(-1)).squeeze(-1)
        return scores

    def loss(self, scores, labels):
        return self.loss_fn(scores, labels)

## 5. Data Loading Utilities

In [ ]:
class NewsDataset:
    def __init__(self, newsID_catID, newsID_subcatID, newsID_titleWordID, newsID_abstractWordID,
                 newsID_titleEntity, newsID_abstractEntity):
        self.newsID_catID = newsID_catID
        self.newsID_subcatID = newsID_subcatID
        self.newsID_titleWordID = newsID_titleWordID
        self.newsID_abstractWordID = newsID_abstractWordID
        self.newsID_titleEntity = newsID_titleEntity
        self.newsID_abstractEntity = newsID_abstractEntity

    def get_news_features(self, news_ids, max_title=30, max_abstract=50, max_entity=10):
        batch_size = len(news_ids)

        cats = torch.zeros(batch_size, dtype=torch.long)
        subcats = torch.zeros(batch_size, dtype=torch.long)
        titles = torch.zeros(batch_size, max_title, dtype=torch.long)
        title_masks = torch.zeros(batch_size, max_title)
        abstracts = torch.zeros(batch_size, max_abstract, dtype=torch.long)
        abstract_masks = torch.zeros(batch_size, max_abstract)
        title_ents = torch.zeros(batch_size, max_entity, 2)
        title_ent_masks = torch.zeros(batch_size, max_entity)
        abstract_ents = torch.zeros(batch_size, max_entity, 2)
        abstract_ent_masks = torch.zeros(batch_size, max_entity)

        for i, nid in enumerate(news_ids):
            cats[i] = self.newsID_catID.get(nid, 0)
            subcats[i] = self.newsID_subcatID.get(nid, 0)

            title_words = self.newsID_titleWordID.get(nid, [])
            title_len = min(len(title_words), max_title)
            titles[i, :title_len] = torch.tensor(title_words[:title_len])
            title_masks[i, :title_len] = 1

            abstract_words = self.newsID_abstractWordID.get(nid, [])
            abstract_len = min(len(abstract_words), max_abstract)
            abstracts[i, :abstract_len] = torch.tensor(abstract_words[:abstract_len])
            abstract_masks[i, :abstract_len] = 1

            title_entities = self.newsID_titleEntity.get(nid, [])
            te_len = min(len(title_entities), max_entity)
            for j, te in enumerate(title_entities[:te_len]):
                title_ents[i, j] = torch.tensor(te)
                title_ent_masks[i, j] = 1

            abstract_entities = self.newsID_abstractEntity.get(nid, [])
            ae_len = min(len(abstract_entities), max_entity)
            for j, ae in enumerate(abstract_entities[:ae_len]):
                abstract_ents[i, j] = torch.tensor(ae)
                abstract_ent_masks[i, j] = 1

        return cats, subcats, titles, title_masks, abstracts, abstract_masks, title_ents, title_ent_masks, abstract_ents, abstract_ent_masks

news_dataset = NewsDataset(newsID_catID, newsID_subcatID, newsID_titleWordID, newsID_abstractWordID,
                           newsID_titleEntity, newsID_abstractEntity)

## 6. Training

In [ ]:
# Initialize model
news_encoder = NewsEncoder(num_cat, num_subcat, title_embed_matrix, abstract_embed_matrix,
                           entity_embed_matrix, config).to(device)
model = NRMS(news_encoder, config).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'], weight_decay = 1e-5)

print(f'Model initialized with {sum(p.numel() for p in model.parameters())} parameters')

Model initialized with 30139972 parameters


In [16]:
# Pre-compute all news embeddings (Option 3 - major speedup)
print('Pre-computing all news embeddings...')
model.eval()

all_news_ids = list(set(newsID_titleWordID.keys()) | set(newsID_abstractWordID.keys()))
print(f'Total news articles: {len(all_news_ids)}')

news_embeddings = {}
batch_size_precompute = 512

with torch.no_grad():
    for i in tqdm(range(0, len(all_news_ids), batch_size_precompute), desc='Pre-computing'):
        batch_ids = all_news_ids[i:i+batch_size_precompute]
        features = news_dataset.get_news_features(batch_ids)
        features = [f.to(device) for f in features]
        embeds = model.news_encoder(*features)
        for nid, emb in zip(batch_ids, embeds):
            news_embeddings[nid] = emb.detach()

# Add PAD embedding
pad_embed = torch.zeros(config['news_final_embed_size']).to(device)
news_embeddings['PAD'] = pad_embed
print(f'Pre-computed {len(news_embeddings)} news embeddings')

Pre-computing all news embeddings...
Total news articles: 65238


Pre-computing: 100%|██████████| 128/128 [00:08<00:00, 15.85it/s]

Pre-computed 65239 news embeddings


In [ ]:
def train_epoch_fast(model, data, news_embeddings, optimizer, device, batch_size=64):
    """Fast training using pre-computed embeddings with true batching."""
    model.train()
    total_loss = 0
    n_batches = 0

    indices = list(range(len(data)))
    random.shuffle(indices)
    num_candidates = config['negative_sampling_num'] + 1  # K+1 candidates

    for i in tqdm(range(0, len(indices), batch_size), desc='Training'):
        batch_indices = indices[i:i+batch_size]
        batch_data = [data[j] for j in batch_indices]
        actual_batch_size = len(batch_data)

        max_hist = max(len(sample[1]) if sample[1] else 1 for sample in batch_data)

        batch_history = torch.zeros(actual_batch_size, max_hist, config['news_final_embed_size']).to(device)
        history_masks = torch.zeros(actual_batch_size, max_hist).to(device)
        batch_candidates = torch.zeros(actual_batch_size, num_candidates, config['news_final_embed_size']).to(device)

        for b, sample in enumerate(batch_data):
            history = sample[1] if sample[1] else ['PAD']
            for h, nid in enumerate(history):
                if nid in news_embeddings:
                    batch_history[b, h] = news_embeddings[nid]
                    history_masks[b, h] = 1

            for c, nid in enumerate(sample[3][:num_candidates]):
                if nid in news_embeddings:
                    batch_candidates[b, c] = news_embeddings[nid]

        optimizer.zero_grad()
        scores = model(batch_history, history_masks, batch_candidates)
        labels = torch.zeros(actual_batch_size, dtype=torch.long).to(device)
        loss = model.loss(scores, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / n_batches


def evaluate_fast(model, data, news_embeddings, device, batch_size=64):
    """Fast evaluation using pre-computed embeddings."""
    model.eval()
    total_loss = 0
    n_batches = 0
    num_candidates = config['negative_sampling_num'] + 1

    with torch.no_grad():
        for i in tqdm(range(0, len(data), batch_size), desc='Evaluating'):
            batch_data = data[i:i+batch_size]
            actual_batch_size = len(batch_data)

            max_hist = max(len(sample[1]) if sample[1] else 1 for sample in batch_data)

            batch_history = torch.zeros(actual_batch_size, max_hist, config['news_final_embed_size']).to(device)
            history_masks = torch.zeros(actual_batch_size, max_hist).to(device)
            batch_candidates = torch.zeros(actual_batch_size, num_candidates, config['news_final_embed_size']).to(device)

            for b, sample in enumerate(batch_data):
                history = sample[1] if sample[1] else ['PAD']
                for h, nid in enumerate(history):
                    if nid in news_embeddings:
                        batch_history[b, h] = news_embeddings[nid]
                        history_masks[b, h] = 1

                for c, nid in enumerate(sample[3][:num_candidates]):
                    if nid in news_embeddings:
                        batch_candidates[b, c] = news_embeddings[nid]

            scores = model(batch_history, history_masks, batch_candidates)
            labels = torch.zeros(actual_batch_size, dtype=torch.long).to(device)
            loss = model.loss(scores, labels)

            total_loss += loss.item()
            n_batches += 1

    return total_loss / n_batches

In [ ]:
# Training loop with FAST pre-computed embeddings
train_losses = []
val_losses = []
best_val_loss = float('inf')
patience = 5
patience_counter = 0

print(f'Training on {len(training_data)} samples, validating on {len(validation_data)} samples')
print('Using FAST training with pre-computed embeddings + K=4 sampling')

for epoch in range(config['epochs']):
    print(f'\n=== Epoch {epoch + 1}/{config["epochs"]} ===')

    train_loss = train_epoch_fast(model, training_data, news_embeddings, optimizer, device, config['batch_size'])
    train_losses.append(train_loss)
    print(f'Train Loss: {train_loss:.4f}')

    val_loss = evaluate_fast(model, validation_data, news_embeddings, device, config['batch_size'])
    val_losses.append(val_loss)
    print(f'Val Loss: {val_loss:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'best_model.pt'))
        print('Model saved!')
    else:
        patience_counter += 1
        print(f'No improvement. Patience: {patience_counter}/{patience}')

    # Early stopping
    if patience_counter >= patience:
        print(f'Early stopping triggered after {epoch + 1} epochs')
        break

print('\nTraining complete!')

Training on 236344 samples, validating on 111383 samples
Using FAST training with pre-computed embeddings + K=4 sampling

=== Epoch 1/40 ===


Training: 100%|██████████| 3693/3693 [03:30<00:00, 17.56it/s]


Train Loss: 1.5305


Evaluating: 100%|██████████| 1741/1741 [01:32<00:00, 18.74it/s]


Val Loss: 1.5347
Model saved!

=== Epoch 2/40 ===


...

Training: 100%|██████████| 3693/3693 [03:32<00:00, 17.36it/s]


Train Loss: 1.4944


Evaluating: 100%|██████████| 1741/1741 [01:33<00:00, 18.65it/s]


Val Loss: 1.5217
No improvement. Patience: 1/5

=== Epoch 9/40 ===


Training:  10%|▉         | 366/3693 [00:20<03:13, 17.17it/s]

## 7. Visualization

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss', marker='o')
plt.plot(val_losses, label='Val Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('NRMS Training Progress')
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curve.png'), dpi=150)
plt.show()

## 8. Testing & Evaluation Metrics

Evaluating with NRMS paper metrics (EMNLP 2019): AUC, MRR, nDCG@5, nDCG@10

In [ ]:
from sklearn.metrics import roc_auc_score

def dcg_score(y_true, y_score, k=10):
    order = np.argsort(y_score)[::-1]
    y_true = np.take(y_true, order[:k])
    gains = 2 ** y_true - 1
    discounts = np.log2(np.arange(len(y_true)) + 2)
    return np.sum(gains / discounts)

def ndcg_score(y_true, y_score, k=10):
    best = dcg_score(y_true, y_true, k)
    actual = dcg_score(y_true, y_score, k)
    return actual / best if best > 0 else 0.0

def mrr_score(y_true, y_score):
    order = np.argsort(y_score)[::-1]
    y_true = np.take(y_true, order)
    for i, val in enumerate(y_true):
        if val == 1:
            return 1.0 / (i + 1)
    return 0.0

print('Metric functions defined')

In [ ]:
def prepare_test_data(behaviors):
    test_samples = []
    for _, row in tqdm(behaviors.iterrows(), total=len(behaviors), desc='Preparing test data'):
        history = str(row['Histories']).split() if pd.notna(row['Histories']) else []
        impressions = str(row['Impressions']).split() if pd.notna(row['Impressions']) else []
        if len(impressions) == 0: continue
        candidates, labels = [], []
        for imp in impressions:
            parts = imp.split('-')
            if len(parts) == 2:
                candidates.append(parts[0])
                labels.append(int(parts[1]))
        if len(candidates) > 0 and sum(labels) > 0:
            test_samples.append({'history': history[-config['max_history_len']:], 'candidates': candidates, 'labels': labels})
    return test_samples

test_data = prepare_test_data(behaviors_dev)
print(f'Test samples: {len(test_data)}')

In [ ]:
def test_model(model, test_data, news_embeddings, device):
    model.eval()
    all_auc, all_mrr, all_ndcg5, all_ndcg10 = [], [], [], []

    with torch.no_grad():
        for sample in tqdm(test_data, desc='Testing'):
            history = sample['history'] if sample['history'] else ['PAD']
            candidates = sample['candidates']
            labels = np.array(sample['labels'])

            # Build history embeddings
            max_hist = len(history)
            hist_embed = torch.zeros(1, max_hist, config['news_final_embed_size']).to(device)
            hist_mask = torch.zeros(1, max_hist).to(device)
            for h, nid in enumerate(history):
                if nid in news_embeddings:
                    hist_embed[0, h] = news_embeddings[nid]
                    hist_mask[0, h] = 1

            # Build candidate embeddings
            cand_embed = torch.zeros(1, len(candidates), config['news_final_embed_size']).to(device)
            for c, nid in enumerate(candidates):
                if nid in news_embeddings:
                    cand_embed[0, c] = news_embeddings[nid]

            scores = model(hist_embed, hist_mask, cand_embed).squeeze(0).cpu().numpy()

            try:
                if len(np.unique(labels)) > 1:
                    all_auc.append(roc_auc_score(labels, scores))
            except: pass
            all_mrr.append(mrr_score(labels, scores))
            all_ndcg5.append(ndcg_score(labels, scores, k=5))
            all_ndcg10.append(ndcg_score(labels, scores, k=10))

    return {'AUC': np.mean(all_auc) if all_auc else 0, 'MRR': np.mean(all_mrr), 'nDCG@5': np.mean(all_ndcg5), 'nDCG@10': np.mean(all_ndcg10)}

In [ ]:
# Load best model and run evaluation
best_path = os.path.join(OUTPUT_DIR, 'best_model.pt')
if os.path.exists(best_path):
    model.load_state_dict(torch.load(best_path))
    print('Loaded best model')

results = test_model(model, test_data, news_embeddings, device)

print('\n' + '='*50)
print('TEST RESULTS')
print('='*50)
for m, v in results.items(): print(f'{m}: {v:.4f}')

In [ ]:
# Compare with paper results
paper = {'AUC': 0.6785, 'MRR': 0.3340, 'nDCG@5': 0.3676, 'nDCG@10': 0.4300}

print('\n' + '='*60)
print('COMPARISON WITH NRMS PAPER (EMNLP 2019)')
print('='*60)
print(f'{"Metric":<12}{"Ours":<12}{"Paper":<12}{"Diff":<12}')
print('-'*60)
for m in paper:
    diff = results[m] - paper[m]
    print(f'{m:<12}{results[m]:<12.4f}{paper[m]:<12.4f}{diff:+.4f}')
print('='*60)

In [ ]:
# Metrics comparison chart
metrics = list(paper.keys())
ours = [results[m] for m in metrics]
theirs = [paper[m] for m in metrics]

x = np.arange(len(metrics))
fig, ax = plt.subplots(figsize=(10, 6))
b1 = ax.bar(x - 0.175, ours, 0.35, label='Our Results', color='steelblue')
b2 = ax.bar(x + 0.175, theirs, 0.35, label='Paper Results', color='coral')

ax.set_ylabel('Score')
ax.set_title('NRMS: Our Results vs Paper (EMNLP 2019)')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

for bar, val in zip(b1, ours): ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}', ha='center', fontsize=9)
for bar, val in zip(b2, theirs): ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'metrics_comparison.png'), dpi=150)
plt.show()

In [ ]:
def evaluate_model_performance(model, data, news_dataset, device, batch_size=64):
    model.eval()

    # Lists to store metrics for every user/sample
    auc_scores = []
    mrr_scores = []
    ndcg5_scores = []
    ndcg10_scores = []

    with torch.no_grad():
        for i in tqdm(range(0, len(data), batch_size), desc='Calculating Metrics'):
            batch_data = data[i:i+batch_size]

            # --- 1. PREPARE DATA (Same as training) ---
            all_history = []
            history_lens = []
            for sample in batch_data:
                history = sample[1][-config['max_history_len']:]
                all_history.extend(history if history else ['PAD'])
                history_lens.append(len(history) if history else 1)

            all_candidates = []
            for sample in batch_data:
                all_candidates.extend(sample[3])

            # --- 2. ENCODE NEWS ---
            history_features = news_dataset.get_news_features(all_history)
            history_features = [f.to(device) for f in history_features]
            history_embeds = model.news_encoder(*history_features)

            candidate_features = news_dataset.get_news_features(all_candidates)
            candidate_features = [f.to(device) for f in candidate_features]
            candidate_embeds = model.news_encoder(*candidate_features)

            # --- 3. RESHAPE & FORWARD PASS ---
            max_hist_len = max(history_lens)
            batch_history_embeds = torch.zeros(len(batch_data), max_hist_len, history_embeds.size(-1)).to(device)
            history_masks = torch.zeros(len(batch_data), max_hist_len).to(device)

            idx = 0
            for b, hlen in enumerate(history_lens):
                batch_history_embeds[b, :hlen] = history_embeds[idx:idx+hlen]
                history_masks[b, :hlen] = 1
                idx += hlen

            num_candidates = len(batch_data[0][3]) # Should be 5 (1 pos + 4 neg)
            batch_candidate_embeds = candidate_embeds.view(len(batch_data), num_candidates, -1)

            # Get Scores: [Batch_Size, 5]
            scores = model(batch_history_embeds, history_masks, batch_candidate_embeds)
            scores = scores.cpu().numpy()

            # --- 4. CALCULATE METRICS ---
            # In your data loader, the 1st item (index 0) is always the positive click
            # So the labels are [1, 0, 0, 0, 0] for every sample
            labels = np.zeros((len(batch_data), num_candidates))
            labels[:, 0] = 1

            for j in range(len(batch_data)):
                y_true = labels[j]
                y_score = scores[j]

                try:
                    auc = roc_auc_score(y_true, y_score)
                    mrr = mrr_score(y_true, y_score)
                    ndcg5 = ndcg_score(y_true, y_score, k=5)
                    ndcg10 = ndcg_score(y_true, y_score, k=10)

                    auc_scores.append(auc)
                    mrr_scores.append(mrr)
                    ndcg5_scores.append(ndcg5)
                    ndcg10_scores.append(ndcg10)
                except ValueError:
                    continue

    # Print Averages
    print(f"\nEvaluation Results:")
    print(f"AUC:      {np.mean(auc_scores):.4f}")
    print(f"MRR:      {np.mean(mrr_scores):.4f}")
    print(f"nDCG@5:   {np.mean(ndcg5_scores):.4f}")
    print(f"nDCG@10:  {np.mean(ndcg10_scores):.4f}")

## 9. Summary

**NRMS Implementation with Optimizations:**
- K=4 negative sampling (as per EMNLP 2019 paper)
- Pre-computed news embeddings (~50x speedup)
- True batched training with batch_size=64

**Evaluation Metrics:**
- AUC, MRR, nDCG@5, nDCG@10

In [ ]:
# 1. Load the best weights
print("Loading best model...")
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'best_model.pt')))

# 2. Run evaluation
# Note: validation_data is the list you created in step 3.3 of your notebook
evaluate_model_performance(model, validation_data, news_dataset, device, batch_size=config['batch_size'])